# BSk24 deformation experiment

This notebook is a small front end to the installed `eos_generation` package. It contains no scientific equations or solver implementation. Edit the single settings cell, then choose **Run All**.

The first pass is always a passive preview: it performs zero scientific solver calls and writes nothing. Review the printed plan and destination. To run that exact plan, change only `EXECUTE_REVIEWED_PLAN` to `True` and choose **Run All** again in the same kernel. Any change to settings, source code, environment, process budget, plan, or destination invalidates the preview.

Scalar and list values are both accepted for amplitudes and deformation geometry. Lists form a Cartesian grid automatically; `EPSILON_MATCH` remains one governed matching anchor per experiment. `CALCULATION="thermodynamics"` stops after thermodynamic gating and reconstruction; `CALCULATION="stellar"` requests the governed full stellar route, including fixed-mass and tidal results. `PRECISION="quick"` changes numerical resolution only—the physical acceptance gates are identical to `"strict"`.

In [ ]:
from eos_generation.notebook import NotebookSettings, get_notebook_session

notebook_session = get_notebook_session()

## Experiment settings

All energy-density values are in MeV fm$^{-3}$ and fixed masses are in solar masses. `EPSILON_MATCH="standard"` uses the governed BSk24 matching anchor; numeric alternatives remain subject to the production model's declared-domain checks. `DIAGNOSTICS="on"` is available only for stellar calculations and adds the governed endpoint radial-support diagnostic. Generated packets are written below the ignored `runs/` directory only after explicit execution.

In [ ]:
# Scalars or lists may be used for amplitudes and geometry.
AMPLITUDES = [-0.01, 0.0, 0.01]
EPSILON_MATCH = "standard"
CENTER = 300.0
WIDTH = 50.0
RAMP_WIDTH = 20.0

CALCULATION = "thermodynamics"  # "thermodynamics" or "stellar"
FIXED_MASSES = [1.4]
PRECISION = "strict"             # "quick" or "strict"
DIAGNOSTICS = "off"             # "off" or "on" (stellar only)

# False = passive preview; True = execute the unchanged reviewed preview.
EXECUTE_REVIEWED_PLAN = False

In [ ]:
settings = NotebookSettings.from_values(
    amplitudes=AMPLITUDES,
    epsilon_match=EPSILON_MATCH,
    center=CENTER,
    width=WIDTH,
    ramp_width=RAMP_WIDTH,
    calculation=CALCULATION,
    fixed_masses=FIXED_MASSES,
    precision=PRECISION,
    diagnostics=DIAGNOSTICS,
)

In [ ]:
notebook_run = notebook_session.prepare(
    settings, record_preview=not EXECUTE_REVIEWED_PLAN
)
print(notebook_run.summary_text())

In [ ]:
experiment_result = notebook_session.execute(
    notebook_run,
    current_settings=settings,
    execute=EXECUTE_REVIEWED_PLAN,
)
if experiment_result is None:
    print("EXECUTE_REVIEWED_PLAN=False: execution remains disabled.")
else:
    from IPython.display import Markdown, display

    print(f"Experiment complete: {notebook_run.output_root}")
    locations = notebook_session.student_view_locations()
    links = []
    for label, path in locations.items():
        relative = path.relative_to(notebook_session.repository_root).as_posix()
        links.append(f"- [{label}]({relative})")
    display(Markdown("## Student result locations\n\n" + "\n".join(links)))

## Interpreting a completed run

Accepted and rejected cases retain their exact thermodynamic status and rejection reason. A rejected raw proposal receives no reconstruction or stellar calculation. The reconstructed state is an effective one-fluid cold barotrope; it does not establish microscopic composition or beta equilibrium. Fixed-mass observables require a true bracket on the successful stable branch, and maximum mass is reported as resolved only when a turning point is bracketed and refined under the selected numerical profile.